# RF-DETR on NVIDIA GPUs — Latency Across Sizes and GPUs

One notebook, one GPU per run. For every detection size it times the end-to-end call (preprocessing, forward, postprocessing, `sv.Detections`) on **one decoded RGB array**, batch 1, 4 and 8, for the fastest route of each package:

| Package | Route | Precision |
|---|---|---|
| `rfdetr` | `model.inference()` with a CUDA graph | fp16 |
| `rfdetr` | static-batch `.trt` engine through `RFDETRInference`, `include_source_image=False` | fp16 |
| `rfdetr` | the same, INT8 (sizes in `INT8_SIZES` only) | int8 |
| `inference-models` | managed TensorRT engine | auto |

Run it once on each GPU, then download `gpu_results/<gpu>.json` and put the files from every machine in one `gpu_results/` folder next to a copy of this notebook. The last section merges every JSON it finds there. Colab offers the T4, L4, A100 and RTX PRO 6000 runtimes (Runtime → Change runtime type), which are the four GPUs of the [TensorRT export page](https://rfdetr.roboflow.com/latest/exports/tensorrt/#latency-across-nvidia-gpus).

Latency only: accuracy is scored by the `export-cuda` cookbook (Sections 7 and 8.1), whose engine-only INT8 figures are not comparable with the end-to-end times here. A row that fails (out of memory, a model the package does not publish) is recorded with its error and the run continues. `inference-models` could not select a managed engine for any size on the A100 and RTX PRO 6000 in the published run (`NoModelPackagesAvailableError`), so those rows are errors, not timings.

## 1. Install

In [ ]:
RFDETR_SPEC = "rfdetr[onnx,tensorrt] @ https://github.com/roboflow/rf-detr/archive/refs/heads/develop.zip"
!pip install -q "{RFDETR_SPEC}" "inference-models[trt10]" supervision pandas tabulate faster-coco-eval
!pip install -q --force-reinstall --no-deps "pillow==11.3.0"

## 2. Setup

In [ ]:
import gc
import importlib.metadata
import json
import re
import shutil
import urllib.request
from functools import partial
from pathlib import Path

import numpy as np
import supervision as sv
import tensorrt
import torch
from PIL import Image

from rfdetr import RFDETRInference, RFDETRLarge, RFDETRMedium, RFDETRNano, RFDETRSmall
from rfdetr.export._benchmark import fetch_coco_val2017
from rfdetr.export.benchmark import measure_latency

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a CUDA GPU; none is available.")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_SLUG = re.sub(r"[^a-z0-9]+", "-", GPU_NAME.lower()).strip("-")
WORK_DIR = Path("export_cuda_gpus") / GPU_SLUG  # engines are GPU-specific, so one folder per GPU
RESULTS_DIR = Path("gpu_results")
for folder in (WORK_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

#: size name -> (rfdetr class, `inference-models` model id). Add the sized segmentation classes (`RFDETRSegNano` and up, ids `rfdetr-seg-*`) to measure them too.
MODELS = {
    "nano": (RFDETRNano, "rfdetr-nano"),
    "small": (RFDETRSmall, "rfdetr-small"),
    "medium": (RFDETRMedium, "rfdetr-medium"),
    "large": (RFDETRLarge, "rfdetr-large"),
}
#: INT8 attention runs only up to 325 tokens per window; Large's 485 ran no faster and lost AP (see the placement
#: rules on the TensorRT export page), so Large is left out.
INT8_SIZES = ("nano", "small", "medium")
BATCH_SIZES = (1, 4, 8)
WARMUP_RUNS, MEASURE_RUNS = 20, 100
N_CALIB = 128
CALIBRATION_SEED = 1

versions = {
    "gpu": GPU_NAME,
    "compute_capability": ".".join(map(str, torch.cuda.get_device_capability(0))),
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "tensorrt": tensorrt.__version__,
    "rfdetr": importlib.metadata.version("rfdetr"),
    "inference-models": importlib.metadata.version("inference-models"),
}
print(json.dumps(versions, indent=2))

## 3. Sample image and calibration set

In [ ]:
IMAGE_PATH = WORK_DIR / "sample.jpg"
if not IMAGE_PATH.exists():
    urllib.request.urlretrieve("https://media.roboflow.com/notebooks/examples/dog.jpeg", IMAGE_PATH)
# A camera or video decoder delivers decoded arrays, so every timed call takes this array: no PIL copy, no JPEG decode.
frame = np.array(Image.open(IMAGE_PATH).convert("RGB"))

CALIBRATION_DIR = WORK_DIR / "calibration"
if not CALIBRATION_DIR.exists() or len(list(CALIBRATION_DIR.glob("*.jpg"))) < N_CALIB:
    shutil.rmtree(CALIBRATION_DIR, ignore_errors=True)
    CALIBRATION_DIR.mkdir()
    pool = fetch_coco_val2017(Path("coco"), n_images=N_CALIB, seed=CALIBRATION_SEED)
    for image_id in pool.image_ids[:N_CALIB]:
        shutil.copy(pool.images_dir / f"{image_id:012d}.jpg", CALIBRATION_DIR)
print(f"frame {frame.shape}, {len(list(CALIBRATION_DIR.glob('*.jpg')))} calibration images")

## 4. Measure

In [ ]:
from inference_models import AutoModel, BackendType

CONFIDENCE_THRESHOLD = 0.5
ROWS: list[dict] = []


def im_predict(im_model, frames: list[np.ndarray]) -> list[sv.Detections]:
    """Run an `inference-models` model on RGB arrays and return `sv.Detections`, like `RFDETRInference.predict`."""
    return [p.to_supervision() for p in im_model(frames, confidence=CONFIDENCE_THRESHOLD, input_color_format="rgb")]


def measure_row(size: str, package: str, runtime: str, precision: str, batch: int, make_call) -> None:
    """Build the callable with `make_call`, time it and append one row; a failure becomes a row with an `error`."""
    row = {
        "gpu": GPU_NAME,
        "size": size,
        "package": package,
        "runtime": runtime,
        "precision": precision,
        "batch": batch,
    }
    call = None
    try:
        call = make_call()
        call()  # first call outside the timing: allocations, graph capture, lazy kernels
        result = measure_latency(
            call,
            label=f"{size} {package} {runtime} {precision} b{batch}",
            device="cuda",
            warmup=WARMUP_RUNS,
            runs=MEASURE_RUNS,
        )
        row |= {"mean_ms": result.mean_ms, "std_ms": result.std_ms, "img_per_s": batch * result.fps}
        print(
            f"  {package:16} {runtime:22} {precision:5} b{batch}: {result.mean_ms:7.2f} ms ± {result.std_ms:5.2f}"
            f"  ({row['img_per_s']:6.1f} img/s)",
            flush=True,
        )
    except Exception as error:
        row["error"] = f"{type(error).__name__}: {error}"[:300]
        print(f"  {package:16} {runtime:22} {precision:5} b{batch}: FAILED {row['error']}", flush=True)
    finally:
        del call
        gc.collect()
        torch.cuda.empty_cache()
    ROWS.append(row)


def engine_path(model, size: str, batch: int, precision: str) -> Path:
    """The static-batch engine of *size*, built once per GPU and reused on a rerun."""
    out_dir = WORK_DIR / "engines" / size / f"b{batch}" / precision
    built = sorted(out_dir.glob("*.trt")) if out_dir.exists() else []
    if built:
        return built[0]
    options = (
        {"fp16": True}
        if precision == "FP16"
        else {"quantization": "int8", "calibration_data": str(CALIBRATION_DIR), "max_images": N_CALIB}
    )
    path = Path(model.export(output_dir=str(out_dir), format="tensorrt", batch_size=batch, verbose=False, **options))
    if precision == "INT8":
        assert path.name.endswith("_int8.trt"), f"Expected an INT8 engine, got {path}"
    return path


def trt_call(model, size: str, batch: int, precision: str):
    predictor = RFDETRInference(engine_path(model, size, batch, precision), device="cuda")
    return partial(predictor.predict, [frame] * batch, include_source_image=False)


for size, (model_class, im_id) in MODELS.items():
    print(f"=== {size} ===", flush=True)
    model = model_class()
    for batch in BATCH_SIZES:
        frames = [frame] * batch
        measure_row(
            size,
            "rfdetr",
            "inference() CUDA Graph",
            "fp16",
            batch,
            lambda: partial(
                model.inference(compile_backend="cudagraph", batch_size=batch, dtype=torch.float16).predict,
                frames,
                include_source_image=False,
            ),
        )
        measure_row(size, "rfdetr", ".trt engine", "fp16", batch, lambda: trt_call(model, size, batch, "FP16"))
        if size in INT8_SIZES:
            measure_row(size, "rfdetr", ".trt engine", "int8", batch, lambda: trt_call(model, size, batch, "INT8"))
    del model
    gc.collect()
    torch.cuda.empty_cache()

    try:
        im_model = AutoModel.from_pretrained(im_id, backend=BackendType.TRT)
    except Exception as error:
        for batch in BATCH_SIZES:
            ROWS.append(
                {
                    "gpu": GPU_NAME,
                    "size": size,
                    "package": "inference-models",
                    "runtime": "managed engine",
                    "precision": "auto",
                    "batch": batch,
                    "error": f"{type(error).__name__}: {error}"[:300],
                }
            )
        print(f"  inference-models {im_id}: FAILED to load: {error}", flush=True)
        continue
    for batch in BATCH_SIZES:
        measure_row(
            size,
            "inference-models",
            "managed engine",
            "auto",
            batch,
            lambda: partial(im_predict, im_model, [frame] * batch),
        )
    del im_model
    gc.collect()
    torch.cuda.empty_cache()

## 5. Save this GPU's results

In [ ]:
RESULT_PATH = RESULTS_DIR / f"{GPU_SLUG}.json"
payload = {
    "versions": versions,
    "settings": {
        "warmup": WARMUP_RUNS,
        "runs": MEASURE_RUNS,
        "batches": BATCH_SIZES,
        "confidence": CONFIDENCE_THRESHOLD,
        "input": "one decoded RGB uint8 array, include_source_image=False for rfdetr",
        "int8_sizes": INT8_SIZES,
    },
    "rows": ROWS,
}
RESULT_PATH.write_text(json.dumps(payload, indent=2))
failed = [row for row in ROWS if "error" in row]
print(f"Saved {len(ROWS) - len(failed)} rows and {len(failed)} failures to {RESULT_PATH}")

## 6. Merge every GPU

Reads every `gpu_results/*.json` in the working folder, so run it after copying the files from each machine. Shows the fastest configuration per GPU, size and batch, then the full latency table.

In [ ]:
import pandas as pd

frames_by_gpu = []
for path in sorted(RESULTS_DIR.glob("*.json")):
    data = json.loads(path.read_text())
    table = pd.DataFrame(data["rows"])
    table["tensorrt"] = data["versions"]["tensorrt"]
    frames_by_gpu.append(table)
results = pd.concat(frames_by_gpu, ignore_index=True)
if "error" in results:
    print("Failed rows:")
    print(
        results[results["error"].notna()][
            ["gpu", "size", "package", "runtime", "precision", "batch", "error"]
        ].to_string(index=False)
    )
    results = results[results["error"].isna()]
results["config"] = results["package"] + " · " + results["runtime"] + " · " + results["precision"]
size_order = list(MODELS)
results["size"] = pd.Categorical(results["size"], size_order, ordered=True)

best = results.loc[results.groupby(["gpu", "size", "batch"], observed=True)["mean_ms"].idxmin()]
print("\nFastest configuration per GPU, size and batch")
print(
    best.sort_values(["gpu", "size", "batch"])[["gpu", "size", "batch", "config", "mean_ms", "img_per_s"]].to_markdown(
        index=False, floatfmt=".1f"
    )
)

print("\nBatch-1 end-to-end latency [ms]")
b1 = results[results["batch"] == 1].pivot_table(
    index=["gpu", "size"], columns="config", values="mean_ms", observed=True
)
print(b1.to_markdown(floatfmt=".2f"))

print("\nThroughput at batch 8 [img/s]")
b8 = results[results["batch"] == 8].pivot_table(
    index=["gpu", "size"], columns="config", values="img_per_s", observed=True
)
print(b8.to_markdown(floatfmt=".1f"))